# Pantanal MODIS backfill cleaning, 2003–2012

This notebook turns the nine FIRMS Terra/Aqua archives into cleaned detections, active grid-cell days, and monthly counts. It uses the same Pantanal area, local dates, fire filters, and 2 km grid as the six-season overlap notebook. The backfill ends on January 31, 2012.

**Kaggle.** Open a notebook, use **File → Import Notebook** to import this file, and use **Add Input** to attach a dataset containing the nine ZIPs. Keep their names. Flat files, season folders, and one outer ZIP containing the nine files all work. The notebook also recognizes the original FIRMS CSV names if the ZIPs were unpacked. Select **Run All**. The default output is `/kaggle/working/pantanal_modis_backfill.zip`.

## 1. Configure paths

Use the default Kaggle input location or set `INPUT_PATH` to a specific attached dataset, a local `backfill` directory, or an outer ZIP. `OUTPUT_ZIP` sets the path of the single archive. Each season has a folder inside it.

In [13]:
import hashlib
import io
import tempfile
from contextlib import ExitStack
from pathlib import Path, PurePosixPath
from zipfile import ZIP_DEFLATED, ZipFile
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

# Examples: "/kaggle/input/datasets/your-name/pantanal-modis" or "backfill".
INPUT_PATH = None
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
OUTPUT_ZIP = (KAGGLE_WORKING if KAGGLE_WORKING.is_dir() else Path.cwd()) / "pantanal_modis_backfill.zip"

## 2. Set the analysis scope

Each season runs from February 1 through January 31 in `America/Cuiaba`. The combined MODIS metric uses the union of Terra and Aqua detections by local day and 2 km grid cell. The ZIPs contain detections. Observation coverage stays unverified.

In [14]:
YEAR_STARTS = range(2003, 2012)
SOURCE_IDS = (847975, 847976, 847977, 847978, 847979, 847585, 847587, 847592, 847597)
ZIP_NAMES = {year: f"MODIS {year}-{year + 1}.zip" for year in YEAR_STARTS}
CSV_NAMES = {year: f"fire_archive_M-C61_{source_id}.csv"
             for year, source_id in zip(YEAR_STARTS, SOURCE_IDS)}
ZONE = ZoneInfo("America/Cuiaba")
START = pd.Timestamp(f"{YEAR_STARTS[0]}-02-01")
STOP = pd.Timestamp(f"{YEAR_STARTS[-1] + 1}-02-01")  # Exclusive end.
BOUNDS = (-58, -19.5, -56, -17)  # West, south, east, north.
GRID_CRS = "EPSG:6933"
CELL_SIZE_M = 2_000
METHOD = "pantanal_cell_days_l1_v1"
INPUT_FIELDS = [
    "latitude", "longitude", "brightness", "scan", "track", "acq_date",
    "acq_time", "satellite", "instrument", "confidence", "version",
    "bright_t31", "frp", "daynight", "type",
]
PLATFORM_NAMES = {"A": "Aqua", "AQUA": "Aqua", "T": "Terra", "TERRA": "Terra"}
COUNT_COLUMNS = ["aqua_cell_days", "terra_cell_days", "modis_cell_days"]
CELL_KEYS = ["local_date", "cell_x", "cell_y"]


def season_name(year):
    return f"{year}-{year + 1}"

## 3. Locate the attached data

The inventory looks for the nine season ZIPs first. For Kaggle datasets with unpacked files, it accepts the corresponding FIRMS CSV for each season. If the same archive is present both loose and in its season folder, it uses the season-folder copy when the bytes match. It reports missing or conflicting inputs before writing the output.

In [15]:
def file_name(member):
    return PurePosixPath(member.replace("\\", "/")).name


def register_inputs(paths, kind, indexes, input_base):
    zip_years = {name.casefold(): year for year, name in ZIP_NAMES.items()}
    csv_years = {name.casefold(): year for year, name in CSV_NAMES.items()}
    for container, member in paths:
        name = file_name(member) if member else container.name
        key = name.casefold()
        label = str(container.relative_to(input_base)).replace("\\", "/")
        if member:
            label = f"{label}!{member}"
        if key in zip_years:
            indexes["zip"][zip_years[key]].append((container, member, kind, label))
        if key in csv_years:
            indexes["csv"][csv_years[key]].append((container, member, kind, label))


def locate_inputs(root):
    if not root.exists():
        raise FileNotFoundError(f"Input path does not exist: {root}")
    if root.is_file() and root.suffix.lower() != ".zip":
        raise ValueError("INPUT_PATH must be a directory or a ZIP containing the nine seasons.")

    base = root if root.is_dir() else root.parent
    indexes = {extension: {year: [] for year in YEAR_STARTS} for extension in ("zip", "csv")}
    paths = sorted(p for p in root.rglob("*") if p.is_file()) if root.is_dir() else [root]
    register_inputs(((path, None) for path in paths), "file", indexes, base)

    # Search a bundle only if some seasons are still missing.
    if any(not (indexes["zip"][year] or indexes["csv"][year]) for year in YEAR_STARTS):
        season_zips = {name.casefold() for name in ZIP_NAMES.values()}
        for path in paths:
            if path.suffix.lower() != ".zip" or path.name.casefold() in season_zips:
                continue
            with ZipFile(path) as bundle:
                members = ((path, item.filename) for item in bundle.infolist() if not item.is_dir())
                register_inputs(members, "bundle", indexes, base)

    selected, problems = {}, []
    for year in YEAR_STARTS:
        candidates = indexes["zip"][year] or indexes["csv"][year]
        if len(candidates) > 1 and all(candidate[2] == "file" for candidate in candidates):
            # Prefer the organized copy when the same source is also present loose.
            preferred = min(candidates, key=lambda candidate: candidate[0].parent.name != season_name(year))
            payload = preferred[0].read_bytes()
            if all(candidate[0].read_bytes() == payload for candidate in candidates):
                candidates = [preferred]
        if len(candidates) == 1:
            selected[year] = candidates[0]
        elif not candidates:
            problems.append(f"{season_name(year)}: expected {ZIP_NAMES[year]} or {CSV_NAMES[year]}")
        else:
            paths_found = ", ".join(candidate[3] for candidate in candidates)
            problems.append(f"{season_name(year)}: multiple candidates ({paths_found})")
    if problems:
        raise ValueError("Resolve these inputs and rerun:\n" + "\n".join(problems))
    return selected


if INPUT_PATH is not None:
    input_root = Path(INPUT_PATH).expanduser().resolve()
elif KAGGLE_INPUT.is_dir():
    input_root = KAGGLE_INPUT.resolve()
else:
    input_root = (Path.cwd() / "backfill").resolve()

inputs = locate_inputs(input_root)
print(f"Found {len(inputs)} seasons in {input_root}")
for year, (_, _, _, label) in inputs.items():
    print(f"{season_name(year)}  {label}")

Found 9 seasons in /kaggle/input
2003-2004  datasets/mohamamdalattas/modis-before-2012/MODIS 2003-2004/fire_archive_M-C61_847975.csv
2004-2005  datasets/mohamamdalattas/modis-before-2012/MODIS 2004-2005/fire_archive_M-C61_847976.csv
2005-2006  datasets/mohamamdalattas/modis-before-2012/MODIS 2005-2006/fire_archive_M-C61_847977.csv
2006-2007  datasets/mohamamdalattas/modis-before-2012/MODIS 2006-2007/fire_archive_M-C61_847978.csv
2007-2008  datasets/mohamamdalattas/modis-before-2012/MODIS 2007-2008/fire_archive_M-C61_847979.csv
2008-2009  datasets/mohamamdalattas/modis-before-2012/MODIS 2008-2009/fire_archive_M-C61_847585.csv
2009-2010  datasets/mohamamdalattas/modis-before-2012/MODIS 2009-2010/fire_archive_M-C61_847587.csv
2010-2011  datasets/mohamamdalattas/modis-before-2012/MODIS 2010-2011/fire_archive_M-C61_847592.csv
2011-2012  datasets/mohamamdalattas/modis-before-2012/MODIS 2011-2012/fire_archive_M-C61_847597.csv


## 4. Read and validate the source records

Read the FIRMS CSV directly from each ZIP and preserve the source name and product version. Parse HHMM as UTC and convert it to local time before assigning a season. A February 1 UTC detection can belong to January 31 locally. Reassign such detections to the local season and record the change in the audit.

In [16]:
def read_source(year, source):
    container, member, kind, label = source
    rows_are_zipped = file_name(member or container.name).lower().endswith(".zip")
    with ExitStack() as stack:
        if kind == "bundle":
            bundle = stack.enter_context(ZipFile(container))
            payload = bundle.read(member)
        else:
            payload = container.read_bytes()

        csv_bytes = payload
        source_member = file_name(member or container.name)
        if rows_are_zipped:
            archive = stack.enter_context(ZipFile(io.BytesIO(payload)))
            csv_files = [item for item in archive.infolist()
                         if not item.is_dir() and file_name(item.filename).lower().endswith(".csv")]
            if len(csv_files) != 1:
                raise ValueError(f"{label}: expected exactly one FIRMS CSV, found {len(csv_files)}")
            source_member = csv_files[0].filename
            csv_bytes = archive.read(csv_files[0])

    data = pd.read_csv(io.BytesIO(csv_bytes), dtype=str, keep_default_na=False, encoding="utf-8-sig")
    if set(data.columns) != set(INPUT_FIELDS) or data.empty:
        raise ValueError(f"{label}: unexpected FIRMS columns or no source detections")
    data = data[INPUT_FIELDS].copy()
    platforms = data["satellite"].str.strip().str.upper().map(PLATFORM_NAMES)
    versions = set(data["version"].unique())
    if (set(platforms.dropna()) != {"Aqua", "Terra"} or platforms.isna().any()
            or not data["instrument"].eq("MODIS").all()):
        raise ValueError(f"{label}: expected MODIS detections from both Aqua and Terra")
    if len(versions) != 1 or not versions.issubset({"6.03", "6.1"}):
        raise ValueError(f"{label}: unexpected or mixed MODIS versions {sorted(versions)}")

    hhmm = data["acq_time"].str.zfill(4)
    valid_time = (data["acq_date"].str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}")
                  & data["acq_time"].str.fullmatch(r"[0-9]{1,4}")
                  & hhmm.str[:2].lt("24") & hhmm.str[2:].lt("60"))
    if not valid_time.all():
        raise ValueError(f"{label}: invalid acquisition date or HHMM field")
    acquired_utc = pd.to_datetime(data["acq_date"] + " " + hhmm,
                                  format="%Y-%m-%d %H%M", utc=True, errors="coerce")
    if acquired_utc.isna().any():
        raise ValueError(f"{label}: invalid acquisition date")
    local_date = acquired_utc.dt.tz_convert(ZONE).dt.tz_localize(None).dt.normalize()

    longitude = pd.to_numeric(data["longitude"], errors="coerce")
    latitude = pd.to_numeric(data["latitude"], errors="coerce")
    confidence = pd.to_numeric(data["confidence"], errors="coerce")
    valid_attributes = (longitude.between(-180, 180) & latitude.between(-90, 90)
                        & confidence.between(0, 100) & confidence.mod(1).eq(0)
                        & data["type"].isin(["0", "1", "2", "3"]) & data["daynight"].isin(["D", "N"]))
    if not valid_attributes.all():
        raise ValueError(f"{label}: invalid coordinates, confidence, type, or day/night flag")

    # A valid source row can fall on the adjacent local boundary day.
    nominal_start = pd.Timestamp(f"{year}-02-01")
    nominal_stop = pd.Timestamp(f"{year + 1}-02-01")
    one_day = pd.Timedelta(days=1)
    if not local_date.between(nominal_start - one_day, nominal_stop).all():
        raise ValueError(f"{label}: detections more than one day outside the named season")

    data = data.assign(
        satellite_normalized=platforms, acquired_at_utc=acquired_utc, local_date=local_date,
        source_season=season_name(year), source_file=label, source_member=source_member,
    )
    outside_nominal = local_date.lt(nominal_start) | local_date.ge(nominal_stop)
    audit = {
        "source_season": season_name(year),
        "source_file": label,
        "source_member": source_member,
        "source_payload_sha256": hashlib.sha256(payload).hexdigest(),
        "modis_version": next(iter(versions)),
        "raw_detections": len(data),
        "aqua_raw_detections": int(platforms.eq("Aqua").sum()),
        "terra_raw_detections": int(platforms.eq("Terra").sum()),
        "first_detection_utc": acquired_utc.min(),
        "last_detection_utc": acquired_utc.max(),
        "first_detection_local_date": local_date.min(),
        "last_detection_local_date": local_date.max(),
        "local_season_boundary_rows": int(outside_nominal.sum()),
        "coverage_status": "unverified",
    }
    return data, audit


source_frames, audit_rows = [], []
for year, source in inputs.items():
    frame, audit = read_source(year, source)
    source_frames.append(frame)
    audit_rows.append(audit)

raw = pd.concat(source_frames, ignore_index=True)
source_audit = pd.DataFrame(audit_rows)
if source_audit["modis_version"].nunique() != 1:
    raise ValueError("The sources use different MODIS versions. Review the products before joining them.")
source_audit[["source_season", "modis_version", "raw_detections", "local_season_boundary_rows"]]

,source_season,modis_version,raw_detections,local_season_boundary_rows
0,2003-2004,6.03,1783,0
1,2004-2005,6.03,4732,0
2,2005-2006,6.03,9443,0
3,2006-2007,6.03,1577,0
4,2007-2008,6.03,4586,0
5,2008-2009,6.03,4028,0
6,2009-2010,6.03,5985,0
7,2010-2011,6.03,5912,0
8,2011-2012,6.03,3585,1


## 5. Filter detections and assign grid cells

Remove exact duplicate source rows across archives. Filter to the Pantanal bounds, inferred vegetation fire (`type = 0`), and MODIS confidence of at least 30. Use the same zero-origin grid as the overlap notebook. Changes in apparent activity can reflect satellite sampling and detection conditions.

In [17]:
def grid_indices(longitude, latitude):
    # EPSG:6933, the WGS 84 ellipsoidal cylindrical equal-area projection.
    # Central meridian and false offsets are zero. True-scale latitude is 30°.
    semi_major = 6_378_137.0
    flattening = 1 / 298.257223563
    eccentricity_squared = flattening * (2 - flattening)
    eccentricity = np.sqrt(eccentricity_squared)
    standard_parallel = np.deg2rad(30.0)
    scale = np.cos(standard_parallel) / np.sqrt(
        1 - eccentricity_squared * np.sin(standard_parallel) ** 2
    )
    sin_latitude = np.sin(np.deg2rad(latitude))
    e_sin_latitude = eccentricity * sin_latitude
    authalic_q = (1 - eccentricity_squared) * (
        sin_latitude / (1 - e_sin_latitude ** 2)
        + np.arctanh(e_sin_latitude) / eccentricity
    )
    east_m = semi_major * scale * np.deg2rad(longitude)
    north_m = semi_major * authalic_q / (2 * scale)
    return np.floor(east_m / CELL_SIZE_M).astype("int64"), np.floor(north_m / CELL_SIZE_M).astype("int64")


# Prefer a duplicate copy already filed under its local season for provenance.
local_year = raw["local_date"].dt.year - raw["local_date"].dt.month.eq(1).astype(int)
expected_source = local_year.astype(str) + "-" + (local_year + 1).astype(str)
dedup_order = raw["source_season"].ne(expected_source).sort_values(kind="stable").index
ordered = raw.loc[dedup_order]
duplicate = ordered.duplicated(INPUT_FIELDS)
unique = ordered.loc[~duplicate]
in_window = unique["local_date"].ge(START) & unique["local_date"].lt(STOP)

west, south, east, north = BOUNDS
in_area = (pd.to_numeric(unique["longitude"]).between(west, east)
           & pd.to_numeric(unique["latitude"]).between(south, north))
vegetation = unique["type"].eq("0")
confidence_ok = pd.to_numeric(unique["confidence"]).ge(30)
kept = in_window & in_area & vegetation & confidence_ok

clean = unique.loc[kept].copy()
clean["year"] = clean["local_date"].dt.year
clean["season_start_year"] = local_year.loc[clean.index]
clean["cell_x"], clean["cell_y"] = grid_indices(
    pd.to_numeric(clean["longitude"]).to_numpy(), pd.to_numeric(clean["latitude"]).to_numpy()
)
clean = clean.sort_values(["local_date", "acquired_at_utc", "satellite_normalized", "latitude", "longitude"])

source_audit = source_audit.set_index("source_season")
removed_or_kept = {
    "exact_duplicate_rows_removed": ordered.loc[duplicate],
    "outside_backfill_dates": unique.loc[~in_window],
    "outside_pantanal_bounds": unique.loc[in_window & ~in_area],
    "other_hotspot_types": unique.loc[in_window & in_area & ~vegetation],
    "low_confidence_detections": unique.loc[in_window & in_area & vegetation & ~confidence_ok],
    "retained_detections": clean,
}
for audit_column, records in removed_or_kept.items():
    counts = records["source_season"].value_counts()
    source_audit[audit_column] = counts.reindex(source_audit.index, fill_value=0)
source_audit = source_audit.reset_index()

print(f"{len(raw):,} raw detections to {len(clean):,} qualifying detections")
source_audit[["source_season", *removed_or_kept]]

41,631 raw detections to 38,803 qualifying detections


,source_season,exact_duplicate_rows_removed,outside_backfill_dates,outside_pantanal_bounds,other_hotspot_types,low_confidence_detections,retained_detections
0,2003-2004,0,0,0,3,78,1702
1,2004-2005,0,0,0,9,230,4493
2,2005-2006,0,0,0,10,627,8806
3,2006-2007,0,0,0,6,78,1493
4,2007-2008,0,0,0,9,225,4352
5,2008-2009,0,0,0,7,304,3717
6,2009-2010,0,0,0,3,393,5589
7,2010-2011,0,0,0,11,429,5472
8,2011-2012,0,0,0,7,399,3179


## 6. Count active cell-days and build the monthly calendar

If Terra and Aqua both report the same cell on the same date, the combined count is one for that cell and date. Export both platform counts as well as the combined count. Empty VIIRS columns are compatible with the existing calendar schema and leave space for actual observations or a separately validated estimate.

In [18]:
platform_cells = (
    clean.assign(aqua_detected=clean["satellite_normalized"].eq("Aqua"),
                 terra_detected=clean["satellite_normalized"].eq("Terra"))
    .groupby(CELL_KEYS, as_index=False, sort=True)[["aqua_detected", "terra_detected"]].any()
)
platform_cells["year"] = platform_cells["local_date"].dt.year
platform_cells["season_start_year"] = (
    platform_cells["year"] - platform_cells["local_date"].dt.month.eq(1).astype(int)
)

dates = pd.date_range(START, STOP, freq="D", inclusive="left", name="local_date")
daily = (
    platform_cells.assign(modis_cell_days=1)
    .groupby("local_date")[["aqua_detected", "terra_detected", "modis_cell_days"]]
    .sum().reindex(dates, fill_value=0)
    .rename(columns={"aqua_detected": "aqua_cell_days", "terra_detected": "terra_cell_days"})
    .astype("int64")
)

monthly = daily.groupby(daily.index.to_period("M")).sum()
starts = monthly.index.to_timestamp()
monthly.insert(0, "period_kind", "month")
monthly.insert(1, "period_start_local", starts)
monthly.insert(2, "period_end_local", starts + pd.offsets.MonthEnd())
monthly["year"] = starts.year
monthly["season_start_year"] = starts.year - (starts.month == 1).astype(int)
monthly["snpp_cell_days"] = pd.array([pd.NA] * len(monthly), dtype="Int64")
monthly["calibrated_modis_cell_days"] = pd.array([pd.NA] * len(monthly), dtype="Float64")
monthly["coverage_status"] = "unverified"
monthly["value_source"] = "observed_modis"
monthly["timezone"] = str(ZONE)
monthly["grid_crs"] = GRID_CRS
monthly["cell_size_m"] = CELL_SIZE_M
monthly["method_version"] = METHOD
monthly["modis_version"] = source_audit["modis_version"].iat[0]
monthly = monthly.reset_index(drop=True)
monthly[["period_start_local", *COUNT_COLUMNS, "coverage_status"]].tail(12)

,period_start_local,aqua_cell_days,terra_cell_days,modis_cell_days,coverage_status
96,2011-02-01,8,2,10,unverified
97,2011-03-01,0,0,0,unverified
98,2011-04-01,3,1,4,unverified
99,2011-05-01,3,3,6,unverified
100,2011-06-01,2,10,11,unverified
101,2011-07-01,45,19,63,unverified
102,2011-08-01,71,47,113,unverified
103,2011-09-01,203,110,277,unverified
104,2011-10-01,137,120,229,unverified
105,2011-11-01,306,274,517,unverified


## 7. Verify counts and the February 2012 join

Check the nine expected seasons, one row per month, the daily-to-monthly totals, and the combined count bounds. A zero records zero qualifying detections in the supplied files. It carries the same unverified observation coverage as the rest of the series.

In [19]:
def require(condition, message):
    if not condition:
        raise ValueError(message)


require(len(monthly) == 108, "Expected 108 monthly rows.")
require(monthly["period_start_local"].tolist() == list(pd.date_range(START, STOP, freq="MS", inclusive="left")),
        "The monthly dates are missing, duplicated, or out of order.")
require(monthly["season_start_year"].value_counts().sort_index().to_dict() == dict.fromkeys(YEAR_STARTS, 12),
        "Each season must contain 12 months.")
require(monthly["period_end_local"].iat[-1] + pd.Timedelta(days=1) == STOP,
        "The final month must end the day before the overlap starts.")
require(daily[COUNT_COLUMNS].sum().eq(monthly[COUNT_COLUMNS].sum()).all(),
        "Daily and monthly totals disagree.")
require(daily["modis_cell_days"].ge(daily[COUNT_COLUMNS[:2]].max(axis=1)).all()
        and daily["modis_cell_days"].le(daily[COUNT_COLUMNS[:2]].sum(axis=1)).all(),
        "The combined MODIS count is outside the Terra/Aqua union bounds.")
require(not platform_cells.duplicated(CELL_KEYS).any() and daily["modis_cell_days"].sum() == len(platform_cells),
        "Duplicate cell-days or inconsistent totals.")
require(source_audit[list(removed_or_kept)].sum(axis=1).eq(source_audit["raw_detections"]).all(),
        "The audit counts do not reconcile to the source counts.")
require(monthly[["snpp_cell_days", "calibrated_modis_cell_days"]].isna().all().all(),
        "The backfill must leave VIIRS and calibrated fields blank.")
print(f"Verified {len(monthly)} months, {len(platform_cells):,} active MODIS cell-days.")

Verified 108 months, 26,486 active MODIS cell-days.


## 8. Export all results to one ZIP

The ZIP has one folder per season with cleaned detections, cell-days, and monthly counts. The audit and the website-ready monthly CSV are at the root of the archive. Append its rows to the monthly rows in the six-year export, sorted by `period_start_local`. Keep pre-overlap values in the observed MODIS view until a monthly VIIRS-equivalent adjustment has been validated.

In [20]:
def write_csv(archive, member, frame):
    content = frame.to_csv(index=False, lineterminator="\n", na_rep="")
    archive.writestr(member, content.encode("utf-8"))


output_zip = Path(OUTPUT_ZIP).expanduser().resolve()
if output_zip == input_root or input_root in output_zip.parents:
    raise ValueError("Choose an output ZIP outside INPUT_PATH.")
if KAGGLE_INPUT.is_dir() and output_zip.is_relative_to(KAGGLE_INPUT.resolve()):
    raise ValueError("Write the output ZIP under /kaggle/working.")

output_zip.parent.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(dir=output_zip.parent, suffix=".tmp", delete=False) as stream:
    staging = Path(stream.name)
try:
    with ZipFile(staging, mode="w", compression=ZIP_DEFLATED) as archive:
        for year in YEAR_STARTS:
            season = season_name(year)
            season_clean = clean.loc[clean["season_start_year"].eq(year)]
            season_cells = platform_cells.loc[platform_cells["season_start_year"].eq(year)]
            season_monthly = monthly.loc[monthly["season_start_year"].eq(year)]
            write_csv(archive, f"{season}/modis_cleaned_detections.csv", season_clean)
            write_csv(archive, f"{season}/modis_cell_days.csv", season_cells)
            write_csv(archive, f"{season}/modis_monthly_activity.csv", season_monthly)

        write_csv(archive, "modis_input_audit.csv", source_audit)
        write_csv(archive, "modis_backfill_monthly.csv", monthly)
    staging.replace(output_zip)
finally:
    staging.unlink(missing_ok=True)

print(f"Output ZIP: {output_zip}")

Output ZIP: /kaggle/working/pantanal_modis_backfill.zip
